# ESZA019 — Visão Computacional
## Laboratório 8 — Rastreamento de objetos em vídeo

**Grupo 2**  
**Autores:** Cesar de Jesus, Mariana Chiara e Vinicius de Marchi  
**Docente:** Prof. Celso Setsuo Kurashima  
**Data de realização dos experimentos:** 27 de julho de 2026  
**Data de publicação:** 29 de julho de 2026

---

### Resumo

Neste laboratório foi implementado um sistema de rastreamento visual de curto prazo com Python e OpenCV. Uma região de interesse (*Region of Interest*, ROI) é escolhida manualmente em um quadro inicial e, a partir dela, o rastreador CSRT estima automaticamente a posição do alvo nos quadros seguintes. O programa foi aplicado a dois vídeos gravados — uma trena amarela e uma pessoa em movimento — e também a imagens adquiridas ao vivo por webcam, nas quais o rótulo de uma garrafa foi rastreado.

O rastreamento da pessoa e da garrafa permaneceu consistente durante os respectivos ensaios. No vídeo da trena ocorreu *drift* no trecho final: a caixa continuou sendo atualizada, mas passou a acompanhar uma região próxima à mão em vez do objeto. Esse resultado permitiu discutir de forma experimental a influência da seleção inicial da ROI, do desfoque de movimento, da alteração de aparência e da ausência de um mecanismo de redetecção.


## Sumário

1. Introdução  
2. Objetivos  
3. Fundamentação teórica  
4. Materiais e métodos  
5. Implementação computacional  
6. Resultados  
7. Análise e discussão  
8. Conclusões  
9. Estrutura do repositório  
10. Referências  
11. Declaração de uso de IA


## 1. Introdução

O rastreamento visual tem como objetivo localizar um alvo ao longo de uma sequência de imagens. Diferentemente da detecção de objetos, que procura novamente as categorias de interesse em cada quadro, o rastreamento de curto prazo parte de uma posição inicial conhecida e atualiza a localização do mesmo alvo à medida que o vídeo avança.

Neste experimento, o usuário fornece a posição inicial desenhando um retângulo ao redor do alvo. O algoritmo utiliza a aparência contida nessa região para estimar, em cada novo quadro, a caixa delimitadora mais provável. O procedimento foi implementado para duas fontes: arquivo de vídeo e webcam. Em ambos os modos, o resultado é exibido na tela e gravado em um novo arquivo MP4.

Além de verificar situações de rastreamento bem-sucedido, o laboratório permite observar limitações práticas. Mudanças rápidas de posição, escala ou orientação, oclusões, desfoque e inclusão de partes do fundo na ROI podem fazer a caixa se afastar gradualmente do objeto verdadeiro.


## 2. Objetivos

O objetivo geral foi implementar e avaliar um sistema de rastreamento de objetos em vídeo com seleção manual da região inicial.

Objetivos específicos:

- abrir vídeos gravados e imagens de webcam com o OpenCV;
- permitir que o usuário escolha manualmente a ROI com o mouse;
- inicializar e atualizar o rastreador CSRT;
- desenhar a caixa delimitadora estimada em cada quadro;
- permitir nova seleção da ROI quando necessário;
- salvar os vídeos processados;
- comparar qualitativamente o comportamento em diferentes tipos de alvo e movimento;
- identificar causas de sucesso, perda e *drift* do rastreamento;
- documentar o experimento em um notebook reprodutível.


## 3. Fundamentação teórica

### 3.1 Rastreamento visual

Seja \(I_t\) o quadro de índice \(t\) e seja

\[
B_t=(x_t,y_t,w_t,h_t)
\]

a caixa delimitadora do alvo, descrita pela posição do canto superior esquerdo, largura e altura. Após receber \(B_0\) no quadro inicial, o rastreador estima \(B_t\) para os quadros seguintes. Essa formulação é denominada rastreamento de alvo único e de curto prazo.

O rastreador não conhece semanticamente o objeto. Ele aprende um modelo visual a partir da ROI inicial e procura uma região com resposta semelhante nos quadros posteriores. Por isso, a qualidade e o enquadramento da ROI inicial influenciam diretamente o resultado.

### 3.2 Seleção da ROI

A função `cv2.selectROI` apresenta um quadro e permite desenhar a caixa com o mouse. A seleção é manual apenas no início — ou quando o usuário solicita uma reinicialização pressionando `R`. Depois disso, a atualização ocorre automaticamente.

Uma boa ROI deve:

- conter o alvo por inteiro;
- ser suficientemente justa;
- evitar grande quantidade de fundo;
- conter textura, contornos ou cores capazes de distinguir o alvo da vizinhança.

### 3.3 Rastreador CSRT

Foi escolhido o CSRT, implementação do método **Discriminative Correlation Filter with Channel and Spatial Reliability**. O método utiliza filtros de correlação discriminativos, ponderação de canais de características e uma estimativa de confiabilidade espacial. Em termos práticos, o CSRT tende a apresentar boa precisão e adaptação a mudanças moderadas de escala, embora seja mais lento do que alternativas como MOSSE e KCF.

A confiabilidade espacial procura concentrar o modelo nas partes da região que são mais adequadas ao rastreamento, enquanto a confiabilidade por canal pondera a contribuição de diferentes características. Essas propriedades justificam a escolha para alvos com cor e forma bem definidas, como a trena amarela e o rótulo vermelho da garrafa.

### 3.4 Sucesso computacional e sucesso semântico

O método `update` devolve uma variável booleana e uma nova caixa. Um retorno verdadeiro significa que o algoritmo calculou uma localização válida; não garante que a caixa ainda esteja sobre o objeto correto. Se o modelo se adaptar ao fundo ou a uma região semelhante, pode ocorrer *drift* mesmo com a mensagem `RASTREAMENTO OK`. A inspeção visual continua sendo necessária quando não existe anotação de referência (*ground truth*).


## 4. Materiais e métodos

### 4.1 Materiais

- computador pessoal com Windows;
- câmera integrada/USB;
- trena amarela;
- garrafa com rótulo vermelho;
- Python 3;
- OpenCV com módulos adicionais (`opencv-contrib-python`);
- Visual Studio Code e terminal PowerShell;
- Jupyter Notebook para documentação.

Todos os vídeos foram processados com resolução de **640 × 480 pixels** e gravados em formato MP4.

### 4.2 Dados experimentais

| Arquivo | Origem | Conteúdo | Duração |
|---|---|---|---:|
| `WIN_20260727_20_15_55_Pro.mp4` | vídeo gravado | trena amarela | 20,40 s |
| `WIN_20260727_20_17_15_Pro.mp4` | vídeo gravado | pessoa em movimento | 12,50 s |
| `trena_rastreada.mp4` | resultado | trena com caixa do CSRT | 12,73 s |
| `pessoa_rastreada.mp4` | resultado | pessoa com caixa do CSRT | 8,72 s |
| `webcam_rastreada.mp4` | webcam ao vivo | rótulo da garrafa | 12,13 s |

### 4.3 Fluxo do programa

1. Os argumentos da linha de comando são lidos.
2. O vídeo ou a webcam é aberto por `cv2.VideoCapture`.
3. No modo vídeo, o usuário pode avançar até um quadro adequado.
4. A ROI inicial é selecionada com `cv2.selectROI`.
5. O CSRT é criado e inicializado com o quadro e a ROI.
6. Para cada quadro seguinte, `update` fornece uma nova caixa.
7. A caixa, o estado e o FPS de processamento são desenhados.
8. O quadro processado é gravado por `cv2.VideoWriter`.
9. `R` permite reinicializar a ROI; `Q` ou `Esc` encerra o experimento.

### 4.4 Particularidades dos ensaios

No vídeo da trena, o objeto ainda não estava bem posicionado no primeiro instante. Por isso foi utilizado `--inicio 4`, iniciando a escolha do quadro aproximadamente aos 4 s. No ensaio da pessoa, a ROI foi desenhada ao redor do corpo. No modo webcam, foi selecionado o rótulo da garrafa, e o objeto foi deslocado, aproximado e levemente rotacionado diante da câmera.


## 5. Implementação computacional

O programa completo está disponível em [`rastreamento_lab8.py`](rastreamento_lab8.py). Ele aceita os modos `video` e `webcam`, utiliza CSRT como padrão e também preserva opções para KCF, MIL, MOSSE e GOTURN.

### 5.1 Instalação

No Windows:

```powershell
py -m pip install opencv-contrib-python
```

A distribuição `opencv-contrib-python` é necessária porque contém o módulo de rastreamento usado pelo CSRT.


In [ ]:
# Núcleo simplificado do procedimento implementado em rastreamento_lab8.py.
# A execução interativa completa deve ser feita pelo terminal, pois selectROI
# abre uma janela gráfica para a seleção com o mouse.

import cv2

captura = cv2.VideoCapture("WIN_20260727_20_17_15_Pro.mp4")
sucesso, quadro = captura.read()

roi = cv2.selectROI(
    "Selecione a ROI",
    quadro,
    fromCenter=False,
    showCrosshair=True,
)

rastreador = cv2.TrackerCSRT_create()
rastreador.init(quadro, roi)

while True:
    sucesso, quadro = captura.read()
    if not sucesso:
        break

    rastreado, caixa = rastreador.update(quadro)
    if rastreado:
        x, y, largura, altura = map(int, caixa)
        cv2.rectangle(
            quadro,
            (x, y),
            (x + largura, y + altura),
            (0, 255, 0),
            2,
        )

captura.release()
cv2.destroyAllWindows()


O código final acrescenta escolha do quadro inicial, gravação MP4, cálculo do FPS de processamento, compatibilidade com `cv2.legacy`, mensagens de erro e reinicialização manual da ROI.

### 5.2 Comandos utilizados

**Trena amarela**

```powershell
py rastreamento_lab8.py --modo video --entrada "WIN_20260727_20_15_55_Pro.mp4" --inicio 4 --saida "trena_rastreada.mp4"
```

**Pessoa em movimento**

```powershell
py rastreamento_lab8.py --modo video --entrada "WIN_20260727_20_17_15_Pro.mp4" --saida "pessoa_rastreada.mp4"
```

**Webcam**

```powershell
py rastreamento_lab8.py --modo webcam --camera 0 --saida "webcam_rastreada.mp4"
```

### 5.3 Verificação dos arquivos

A célula abaixo pode ser executada localmente para conferir resolução, FPS, número de quadros e duração.


In [ ]:
from pathlib import Path
import cv2


def metadados_video(caminho):
    captura = cv2.VideoCapture(str(caminho))
    if not captura.isOpened():
        raise RuntimeError(f"Não foi possível abrir: {caminho}")

    largura = int(captura.get(cv2.CAP_PROP_FRAME_WIDTH))
    altura = int(captura.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = float(captura.get(cv2.CAP_PROP_FPS))
    quadros = int(captura.get(cv2.CAP_PROP_FRAME_COUNT))
    captura.release()

    duracao = quadros / fps if fps > 0 else float("nan")
    return {
        "arquivo": Path(caminho).name,
        "resolucao": f"{largura} × {altura}",
        "fps": round(fps, 2),
        "quadros": quadros,
        "duracao_s": round(duracao, 2),
    }


arquivos = [
    "WIN_20260727_20_15_55_Pro.mp4",
    "WIN_20260727_20_17_15_Pro.mp4",
    "trena_rastreada.mp4",
    "pessoa_rastreada.mp4",
    "webcam_rastreada.mp4",
]

for arquivo in arquivos:
    print(metadados_video(arquivo))


Para visualizar os resultados no Jupyter local:


In [ ]:
from IPython.display import Video, display

display(Video("trena_rastreada.mp4", width=640))
display(Video("pessoa_rastreada.mp4", width=640))
display(Video("webcam_rastreada.mp4", width=640))


## 6. Resultados

Os vídeos completos podem ser abertos pelos links:

- [vídeo original da trena](WIN_20260727_20_15_55_Pro.mp4);
- [resultado do rastreamento da trena](trena_rastreada.mp4);
- [vídeo original da pessoa](WIN_20260727_20_17_15_Pro.mp4);
- [resultado do rastreamento da pessoa](pessoa_rastreada.mp4);
- [resultado do rastreamento pela webcam](webcam_rastreada.mp4).

### 6.1 Trena amarela

![Quadros do rastreamento da trena](assets/painel_trena.png)

Nos quadros inicial e intermediário, a caixa acompanha a trena, inclusive durante aproximação e alteração de escala. No trecho final, o movimento rápido produz desfoque e a aparência do alvo muda. A caixa então sofre *drift* para uma região associada à mão e ao fundo.

### 6.2 Pessoa em movimento

![Quadros do rastreamento da pessoa](assets/painel_pessoa.png)

A caixa acompanha o deslocamento lateral da pessoa, a aproximação em relação à câmera e a saída parcial do campo de visão. Não foi observada perda evidente antes do encerramento.

### 6.3 Webcam ao vivo

![Quadros do rastreamento pela webcam](assets/painel_webcam.png)

O CSRT mantém a caixa sobre o rótulo vermelho durante deslocamento, aproximação e rotação moderada da garrafa. O contraste cromático entre o rótulo e o fundo contribuiu para uma representação visual discriminativa.

### 6.4 Síntese qualitativa

| Ensaio | Alvo | Variações principais | Resultado |
|---|---|---|---|
| vídeo 1 | trena amarela | translação, escala e movimento rápido | rastreamento inicial correto; *drift* no final |
| vídeo 2 | pessoa | translação, aproximação e saída parcial | rastreamento mantido |
| webcam | rótulo da garrafa | translação, escala e rotação | rastreamento mantido |


## 7. Análise e discussão

### 7.1 Influência da ROI

No caso da trena, a ROI continha um objeto relativamente pequeno e manipulado pela mão. Durante o movimento, partes da mão e do ambiente passaram a competir com a aparência aprendida. Uma ROI mais justa, com menor quantidade de fundo, tende a reduzir esse efeito, embora não elimine problemas causados por desfoque e oclusão.

A pessoa ocupa uma área maior, com contornos e distribuição de intensidade relativamente estáveis. Mesmo com a aproximação, o CSRT conseguiu adaptar a caixa. Para a garrafa, o rótulo vermelho formou uma região compacta e com bom contraste, facilitando a localização.

### 7.2 *Drift* sem indicação de falha

No último trecho da trena, o texto da interface ainda mostra `RASTREAMENTO OK`, embora a caixa já não represente corretamente o objeto. Isso acontece porque o retorno de `update` informa que foi possível calcular uma caixa, e não que a identidade semântica do alvo foi preservada. Uma avaliação quantitativa exigiria anotações manuais de referência e uma medida como a interseção sobre união (IoU).

### 7.3 FPS da fonte e FPS de processamento

Os vídeos foram gravados próximos de 30 quadros por segundo. O valor exibido na tela pelo programa corresponde à taxa com que o CSRT foi processado no computador e pode ser menor. O `VideoWriter`, contudo, utiliza o FPS da fonte para construir o arquivo de saída. Portanto, a indicação visual de aproximadamente 4–5 FPS nos vídeos do laboratório e 11–12 FPS no ensaio da webcam não representa o FPS nominal do arquivo salvo.

### 7.4 Limitações e melhorias

- o rastreamento depende da seleção manual inicial;
- o CSRT não possui, nesse programa, um detector global para reencontrar o alvo;
- desfoque de movimento, oclusão e saída completa da imagem podem causar perda;
- um retorno de sucesso não detecta automaticamente o *drift*;
- o desempenho computacional depende do hardware e da resolução;
- a tecla `R` fornece recuperação manual, mas interrompe a continuidade automática.

Como melhorias futuras, pode-se combinar detecção e rastreamento, comparar CSRT com KCF/MOSSE/GOTURN, registrar o tempo por quadro e anotar uma sequência para calcular IoU e taxa de sucesso.


## 8. Conclusões

O laboratório atingiu o objetivo de implementar o rastreamento de uma ROI selecionada manualmente tanto em vídeos gravados quanto em imagens ao vivo. O programa abre diferentes fontes, permite escolher o quadro inicial, inicializa o CSRT, desenha a caixa estimada, informa o estado do processamento e salva o resultado.

Os ensaios mostraram que o CSRT foi robusto para a pessoa e para o rótulo da garrafa, mesmo com mudanças de posição, escala e rotação moderada. O caso da trena evidenciou uma limitação importante: após movimento rápido e alteração significativa da aparência, ocorreu *drift* sem que a função `update` sinalizasse falha.

Conclui-se que o CSRT é adequado para rastreamento de curto prazo quando o alvo permanece visualmente distinguível e a ROI inicial é bem selecionada. Para aplicações mais exigentes, especialmente com oclusões ou reaparecimento do objeto, seria necessário associar o rastreador a um detector ou a um mecanismo automático de reinicialização.


## 9. Estrutura do repositório

```text
Lab8_Grupo2/
├── Relatorio_Lab8_Grupo2.ipynb
├── README.md
├── requirements.txt
├── rastreamento_lab8.py
├── WIN_20260727_20_15_55_Pro.mp4
├── WIN_20260727_20_17_15_Pro.mp4
├── trena_rastreada.mp4
├── pessoa_rastreada.mp4
├── webcam_rastreada.mp4
└── assets/
    ├── painel_trena.png
    ├── painel_pessoa.png
    └── painel_webcam.png
```


## 10. Referências

1. KURASHIMA, C. S. **ESZA019 — Visão Computacional: Laboratório 8 — Rastreamento de objetos**. Universidade Federal do ABC, 2026.
2. OPENCV. **cv::TrackerCSRT Class Reference**. Disponível em: <https://docs.opencv.org/4.x/d2/da2/classcv_1_1TrackerCSRT.html>. Acesso em: 29 jul. 2026.
3. OPENCV. **High-level GUI: selectROI**. Disponível em: <https://docs.opencv.org/4.x/d7/dfc/group__highgui.html>. Acesso em: 29 jul. 2026.
4. OPENCV. **cv::VideoCapture Class Reference**. Disponível em: <https://docs.opencv.org/4.x/d8/dfe/classcv_1_1VideoCapture.html>. Acesso em: 29 jul. 2026.
5. OPENCV. **cv::VideoWriter Class Reference**. Disponível em: <https://docs.opencv.org/4.x/dd/d9e/classcv_1_1VideoWriter.html>. Acesso em: 29 jul. 2026.
6. LUKEŽIČ, A.; VOJÍŘ, T.; ČEHOVIN ZAJC, L.; MATAS, J.; KRISTAN, M. **Discriminative Correlation Filter with Channel and Spatial Reliability**. Proceedings of the IEEE Conference on Computer Vision and Pattern Recognition (CVPR), 2017. Disponível em: <https://openaccess.thecvf.com/content_cvpr_2017/html/Lukezic_Discriminative_Correlation_Filter_CVPR_2017_paper.html>.


## 11. Declaração de uso de IA

Ferramentas de inteligência artificial generativa foram utilizadas como apoio na organização do código, na identificação de erros de instalação/importação e na revisão da redação. A gravação dos vídeos, a seleção das regiões de interesse, a execução dos experimentos, a inspeção dos resultados e a validação do conteúdo foram realizadas pelo grupo. Os autores assumem responsabilidade integral pelo material apresentado.
